## Instalando bibliotecas

In [2]:
!pip install transformers datasets peft accelerate torch

## Importando bibliotecas

In [3]:
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)
import torch

## Preparando Dataset

In [ ]:
def convert_to_hf_format(example):
    """Combina instrução e saída em um único texto."""
    return {
        "text": f"Instruction: {example['Instruction']}\nOutput: {example['Output']}"
    }

# Carrega o arquivo JSON Lines
dataset = load_dataset('json', data_files='../dataset.jsonl')
# Aplica a conversão
dataset = dataset.map(convert_to_hf_format)
# Divide em treino e teste
dataset = dataset["train"].train_test_split(test_size=0.2)
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['Instruction', 'Output', 'text'],
        num_rows: 80
    })
    test: Dataset({
        features: ['Instruction', 'Output', 'text'],
        num_rows: 20
    })
})


## Carregar o Modelos Pré-Treinado e o Tokenizador

In [5]:
model_name = "ibm-granite/granite-3.3-2b-instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForCausalLM.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Modelo carregado: {model_name}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

Modelo carregado: ibm-granite/granite-3.3-2b-instruct


## Testando Modelo Crú

In [6]:
def generate_response(model, tokenizer, instruction, input_text=""):
    """Gera uma resposta a partir de uma instrução, usando o modelo fornecido."""
    if input_text:
        prompt = f"Instruction: {instruction}\nInput: {input_text}\nOutput:"
    else:
        prompt = f"Instruction: {instruction}\nOutput:"

    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=150,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,          # ativa amostragem para usar temperatura
        temperature=0.7
    )
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extrai apenas a parte após "Output:"
    resposta = full_output.split("Output:")[-1].strip()
    return resposta

# Exemplo de instrução (deve existir no dataset.jsonl)
test_instruction = "Qual é a prioridade ao inspecionar o local de trabalho antes de usar a máquina?"

print("=== ANTES DO FINE-TUNING ===")
print(f"Instrução: {test_instruction}")
#print(f"Resposta base: {generate_response(base_model, tokenizer, test_instruction)}")

=== ANTES DO FINE-TUNING ===
Instrução: Qual é a prioridade ao inspecionar o local de trabalho antes de usar a máquina?


## Tokenização do Dataset

In [7]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

tokenized_datasets = dataset.map(tokenize_function, batched=True)
print("Dataset tokenizado:", tokenized_datasets)

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Dataset tokenizado: DatasetDict({
    train: Dataset({
        features: ['Instruction', 'Output', 'text', 'input_ids', 'attention_mask'],
        num_rows: 80
    })
    test: Dataset({
        features: ['Instruction', 'Output', 'text', 'input_ids', 'attention_mask'],
        num_rows: 20
    })
})


## Preparar o Modelo para LoRA

In [8]:
# A partir daqui, usaremos uma cópia do modelo base para o fine-tuning
model = base_model  # poderia também ser uma nova instância
model = prepare_model_for_kbit_training(model)

## Verificando Camadas Target

In [9]:
import torch.nn as nn

for name, module in base_model.named_modules():
    if isinstance(module, nn.Linear):
        print(name)

model.layers.0.self_attn.q_proj
model.layers.0.self_attn.k_proj
model.layers.0.self_attn.v_proj
model.layers.0.self_attn.o_proj
model.layers.0.mlp.gate_proj
model.layers.0.mlp.up_proj
model.layers.0.mlp.down_proj
model.layers.1.self_attn.q_proj
model.layers.1.self_attn.k_proj
model.layers.1.self_attn.v_proj
model.layers.1.self_attn.o_proj
model.layers.1.mlp.gate_proj
model.layers.1.mlp.up_proj
model.layers.1.mlp.down_proj
model.layers.2.self_attn.q_proj
model.layers.2.self_attn.k_proj
model.layers.2.self_attn.v_proj
model.layers.2.self_attn.o_proj
model.layers.2.mlp.gate_proj
model.layers.2.mlp.up_proj
model.layers.2.mlp.down_proj
model.layers.3.self_attn.q_proj
model.layers.3.self_attn.k_proj
model.layers.3.self_attn.v_proj
model.layers.3.self_attn.o_proj
model.layers.3.mlp.gate_proj
model.layers.3.mlp.up_proj
model.layers.3.mlp.down_proj
model.layers.4.self_attn.q_proj
model.layers.4.self_attn.k_proj
model.layers.4.self_attn.v_proj
model.layers.4.self_attn.o_proj
model.layers.4.mlp.g

## Configurar e Injetar LoRA

In [10]:
!pip install --upgrade torchao

In [11]:
lora_config = LoraConfig(
    r=16,                    # rank da decomposição
    lora_alpha=32,           # escala alpha
    target_modules=["q_proj", "k_proj", "v_proj"],  # camadas alvo , "o_proj", "gate_proj", "up_proj", "down_proj"
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    inference_mode=False     # False = modo treinamento
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 5,898,240 || all params: 2,539,438,080 || trainable%: 0.2323


## Data Collator (Modelagem Causal)

In [12]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

## Argumentos de Treinamento (Preparando Hiperparâmetros)

In [ ]:
training_args = TrainingArguments(
    output_dir="/modelo_treinado_ibm",          # diretório de saída
    eval_strategy="steps",
    eval_steps=100,
    learning_rate=2e-4,
    per_device_train_batch_size=3,
    per_device_eval_batch_size=3,
    num_train_epochs=10,
    weight_decay=0.01,
    logging_steps=10,
    save_strategy="steps",
    save_steps=100,
    load_best_model_at_end=True,
    report_to="none",               # desabilita logging para wandb/mlflow
)

## Iniciar Treino

In [14]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
)

## Treinar Modelo

In [15]:
trainer.train()

Step,Training Loss,Validation Loss
100,1.300884,2.198982
200,0.648083,2.690772
270,0.564979,2.924257


TrainOutput(global_step=270, training_loss=1.1521674067885788, metrics={'train_runtime': 316.4318, 'train_samples_per_second': 2.528, 'train_steps_per_second': 0.853, 'total_flos': 1498374419251200.0, 'train_loss': 1.1521674067885788, 'epoch': 10.0})

## Salvar o Modelo Ajustado e o Tokenizador

In [ ]:
model.save_pretrained("/lora_ibm_causal_finetuned_model")
tokenizer.save_pretrained("/ibm_tokenizer")

('/content/drive/MyDrive/Colab Notebooks/TreinoIBM/ibm_tokenizer/tokenizer_config.json',
 '/content/drive/MyDrive/Colab Notebooks/TreinoIBM/ibm_tokenizer/chat_template.jinja',
 '/content/drive/MyDrive/Colab Notebooks/TreinoIBM/ibm_tokenizer/tokenizer.json')

## Teste pós fine-tuning

In [ ]:
# Carrega o modelo fine-tunado (apenas os adaptadores LoRA)
finetuned_model = AutoModelForCausalLM.from_pretrained("/lora_ibm_causal_finetuned_model")
finetuned_tokenizer = AutoTokenizer.from_pretrained("/ibm_tokenizer")

# Garante que o token de padding esteja definido
if finetuned_tokenizer.pad_token is None:
    finetuned_tokenizer.pad_token = finetuned_tokenizer.eos_token

Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/240 [00:00<?, ?it/s]

In [18]:
print("=== DEPOIS DO FINE-TUNING ===")
print(f"Instrução: {test_instruction}")
print(f"Resposta ajustada: {generate_response(finetuned_model, finetuned_tokenizer, test_instruction)}")

=== DEPOIS DO FINE-TUNING ===
Instrução: Qual é a prioridade ao inspecionar o local de trabalho antes de usar a máquina?


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Resposta ajustada: Verifique se o local de trabalho está livre de obstruções ou perigos antes de usar a máquina.
